In [4]:
from pathlib import Path
from zipfile import ZipFile
import json

import numpy as np
import pandas as pd

from decoder import decode_trip

def available(*names):
    for name in names:
        if Path(name).is_file():
            return Path(name)
    raise FileNotFoundError(f"Upload {names[0]} to Colab.")

score_file = available("candidate_scores.csv", "baseline_candidate_scores.csv")
ref_file = available("student_test_references.csv", "student_test_references(1).csv")
zip_file = available("Student folder.zip", "Student folder(1).zip", "Student_folder.zip")
scores = pd.read_csv(score_file, dtype={"trip_id": str})
refs = pd.read_csv(ref_file, dtype={"trip_id": str})
if not scores["p_match"].between(0, 1).all():
    raise ValueError("p_match must contain finite classifier probabilities in [0, 1].")
output = Path("pathsnap_decoder")
output.mkdir(exist_ok=True)

def continuity(picks, track, arcs):
    dt = np.diff(track[:, 2])
    eligible = (dt > 0) & (dt <= 60)
    before, after = picks[:-1], picks[1:]
    both = (before >= 0) & (after >= 0)
    linked = np.zeros(len(dt), dtype=bool)
    a, b = before[both], after[both]
    linked[both] = (a == b) | (arcs[a, 1] == arcs[b, 0])
    return int((eligible & linked).sum()), int(eligible.sum())

rows = []
routes = {}
with ZipFile(zip_file) as archive:
    for trip, trip_refs in refs.groupby("trip_id", sort=True):
        prefix = next(
            name.rsplit(".", 1)[0] for name in archive.namelist()
            if name.endswith(f"student_test/{trip}/{trip}.track")
        )
        def read(ext, dtype=float):
            with archive.open(f"{prefix}.{ext}") as handle:
                return np.loadtxt(handle, dtype=dtype, ndmin=2)

        track, nodes, arcs = read("track"), read("nodes"), read("arcs", int)
        candidates = scores.loc[scores.trip_id == trip]
        top = candidates.sort_values("p_match", ascending=False, kind="stable")
        top = top.drop_duplicates("observation_index")
        baseline = np.full(len(track), -1, dtype=int)
        baseline[top.observation_index.to_numpy(dtype=int)] = top.arc_id.to_numpy(dtype=int)

        result = decode_trip(track, nodes, arcs, candidates)
        decoded = np.array([arc if arc is not None else -1
                            for arc in (row["arc_id"] for row in result["matches"])], dtype=int)
        routes[trip] = result

        valid = trip_refs.dropna(subset=["reference_arc_id"])
        index = valid.observation_index.to_numpy(dtype=int)
        truth = valid.reference_arc_id.to_numpy(dtype=int)
        for name, picks in (("Classifier only", baseline), ("Viterbi", decoded)):
            linked, pairs = continuity(picks, track, arcs)
            rows.append({"trip_id": trip, "method": name, "correct": int((picks[index] == truth).sum()),
                         "scored_points": len(index), "linked": linked, "pairs": pairs})
        print(f"Finished {trip}", flush=True)

per_trip = pd.DataFrame(rows)
summary = per_trip.groupby("method")[["correct", "scored_points", "linked", "pairs"]].sum()
summary["top1_pct"] = 100 * summary.correct / summary.scored_points
summary["continuity_pct"] = 100 * summary.linked / summary.pairs
display(summary)
per_trip.to_csv(output / "per_trip.csv", index=False)
summary.to_csv(output / "summary.csv")
(output / "decoded_routes.json").write_text(json.dumps(routes))


Finished 00000000
Finished 00000007
Finished 00000009
Finished 00000040
Finished 00000052
Finished 00000056
Finished 00000065
Finished 00000076
Finished 00000077
Finished 00000091


,correct,scored_points,linked,pairs,top1_pct,continuity_pct
method,,,,,,
Classifier only,16722,19133,17883,19149,87.398735,93.388689
Viterbi,16850,19133,18738,19149,88.067736,97.853674


879929